# Exemplo real de 2025: montar contracts[]

Contrato **00050/2025**, LINEZOLIDA, Ministério da Saúde → **SP HOSPITALAR LTDA**.
O [PNCP](https://pncp.gov.br/app/contratos/00394544000185/2025/950) informa assinatura em **17/02/2025**.
O caso selecionado tem **um contrato e um item**: 1.425 unidades × R$ 13,00 = **R$ 18.525,00**.

Execute as quatro células em ordem, com internet. Os blocos entre aspas triplas explicam cada trecho do código.
O [roteiro de apresentação](roteiro-mapeamento-contratos-exemplo.md) acompanha a demonstração.

In [17]:
#!/usr/bin/env python3
"""
Monta um documento OCDS (release package, JSON) para uma compra do PNCP,
partindo do release OCDS do Compras.gov.br (tender + awards) e agregando o
bloco contracts[] a partir do módulo de contratos, seguindo as rotas da
planilha "Field Level Mapping" (aba (OCDS) 5. Contract).

Fluxo (mesmo do relatório mapeamento_pncp_ocds_contratos_1632):
  1. PNCP /compras            -> idCompra (linkSistemaOrigem) e data de publicação
  2. modulo-ocds/1_releases   -> release com ocid ocds-914jxj-<idCompra>
  3a. modulo-contratos/1_consultarContratos     -> dados do contrato (filtrar idCompra)
  3b. modulo-contratos/2_consultarContratosItem -> itens do contrato (filtra por idCompra)
  Ligação de itens: tender.items.id = idCompra + numeroItem

Uso:
  python pncp_ocds_contracts.py                       # compra 1632/2025, só METFORMINA (item 00007)
  python pncp_ocds_contracts.py --item todos          # compra inteira
  python pncp_ocds_contracts.py --item 00003 --saida dipirona.json

Requer apenas a biblioteca "requests".
"""
import argparse
import json
import re
import sys
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests

PNCP = "https://pncp.gov.br/api/consulta/v1"
COMPRAS = "https://dadosabertos.compras.gov.br"
OCID_PREFIX = "ocds-914jxj-"
TZ_BR = timezone(timedelta(hours=-3))
MOEDA = "BRL"

# Saída padrão: tasks/mapeamento-ocds/output/exemplo-ocds-contracts.json.
# No Jupyter não existe __file__; o diretório de trabalho é o do notebook (docs/).
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUTPUT_DIR = BASE_DIR.parent / "output"
SAIDA_PADRAO = OUTPUT_DIR / "exemplo-ocds-contracts.json"

# Esquema usado para a categoria administrativa do contrato (campo adicional,
# planilha linhas 101-104). Nome local, não é um codelist oficial do OCDS.
ESQUEMA_CATEGORIA = "COMPRASGOV-categoria-contrato"
ESQUEMA_SUBCATEGORIA = "COMPRASGOV-subcategoria-contrato"


# --------------------------------------------------------------------------
# Acesso às APIs
# --------------------------------------------------------------------------
SESSION = requests.Session()


def get_json(url, params=None):
    r = SESSION.get(url, params=params, timeout=180)
    r.raise_for_status()
    return r.json()


def janelas(inicio: date, fim: date, dias=365):
    """As APIs de contratos limitam o intervalo de vigência; quebramos em janelas."""
    a = inicio
    while a <= fim:
        b = min(a + timedelta(days=dias - 1), fim)
        yield a, b
        a = b + timedelta(days=1)


def paginar_compras(url, params):
    """Paginação do módulo de contratos (pagina / tamanhoPagina)."""
    pagina = 1
    while True:
        dados = get_json(url, {**params, "pagina": pagina, "tamanhoPagina": 500})
        yield from dados.get("resultado", [])
        if not dados.get("paginasRestantes"):
            break
        pagina += 1


# --------------------------------------------------------------------------
# Passos 1, 2, 3a, 3b
# --------------------------------------------------------------------------
def buscar_compra_pncp(cnpj, ano, sequencial):
    compra = get_json(f"{PNCP}/orgaos/{cnpj}/compras/{ano}/{sequencial}")
    m = re.search(r"compra=(\d+)", compra.get("linkSistemaOrigem") or "")
    if not m:
        sys.exit("Não foi possível extrair idCompra do linkSistemaOrigem.")
    return compra, m.group(1)


def buscar_release(cnpj, data_pub: str, id_compra):
    """Percorre as páginas do 1_releases até achar o ocid da compra."""
    ocid = OCID_PREFIX + id_compra
    url = f"{COMPRAS}/modulo-ocds/1_releases"
    params = {"buyerID": cnpj, "releaseStartDate": data_pub, "releaseEndDate": data_pub}
    pacote = get_json(url, params)
    while True:
        for rel in pacote.get("releases", []):
            if rel.get("ocid") == ocid:
                return pacote, rel
        prox = (pacote.get("links") or {}).get("next")
        if not prox:
            sys.exit(f"Release {ocid} não encontrado.")
        pacote = get_json(prox)


def buscar_contratos(codigo_orgao, codigo_ug, codigo_modalidade, id_compra, inicio, fim):
    url = f"{COMPRAS}/modulo-contratos/1_consultarContratos"
    achados = {}
    for a, b in janelas(inicio, fim):
        params = {
            "codigoOrgao": codigo_orgao,
            "codigoUnidadeGestora": codigo_ug,
            "codigoModalidadeCompra": codigo_modalidade,
            "dataVigenciaInicialMin": a.isoformat(),
            "dataVigenciaInicialMax": b.isoformat(),
        }
        for c in paginar_compras(url, params):
            if c.get("idCompra") == id_compra:
                achados[c["numeroControlePncpContrato"]] = c
    return list(achados.values())


def buscar_itens_contrato(codigo_orgao, id_compra, inicio, fim):
    url = f"{COMPRAS}/modulo-contratos/2_consultarContratosItem"
    itens = []
    for a, b in janelas(inicio, fim):
        params = {
            "codigoOrgao": codigo_orgao,
            "idCompra": id_compra,
            "dataVigenciaInicialMin": a.isoformat(),
            "dataVigenciaInicialMax": b.isoformat(),
        }
        itens.extend(paginar_compras(url, params))
    return itens


# --------------------------------------------------------------------------
# Utilitários de conversão
# --------------------------------------------------------------------------
def data_br(valor):
    """'2025-11-05T00:00:00' (horário de Brasília, sem fuso) -> ISO com -03:00."""
    if not valor:
        return None
    return datetime.fromisoformat(valor).replace(tzinfo=TZ_BR).isoformat()


def valor(amount, negativo=False):
    if amount is None:
        return None
    amount = round(float(amount), 2)
    return {"amount": -amount if negativo else amount, "currency": MOEDA}


def limpar(obj):
    """Remove chaves com None, listas e objetos vazios (OCDS prefere omitir a usar null)."""
    if isinstance(obj, dict):
        out = {k: limpar(v) for k, v in obj.items()}
        return {k: v for k, v in out.items() if v not in (None, {}, [], "")}
    if isinstance(obj, list):
        out = [limpar(v) for v in obj]
        return [v for v in out if v not in (None, {}, [], "")]
    return obj


# --------------------------------------------------------------------------
# Montagem de contracts[] (rotas da planilha, aba (OCDS) 5. Contract)
# --------------------------------------------------------------------------
def resolver_award_id(release, ni_fornecedor, item_ids):
    """
    contracts/awardID deve apontar para um awards/id existente.
    No release do Compras os awards são agrupados por fornecedor (ids "1".."n"),
    então resolvemos pelo fornecedor (BR-CNPJ-<niFornecedor>) + itens em comum.
    """
    party_id = f"BR-CNPJ-{ni_fornecedor}"
    candidatos = []
    for aw in release.get("awards", []):
        if any(s.get("id") == party_id for s in aw.get("suppliers", [])):
            em_comum = {i.get("id") for i in aw.get("items", [])} & set(item_ids)
            candidatos.append((len(em_comum), aw["id"]))
    if not candidatos:
        return None
    candidatos.sort(reverse=True)
    return candidatos[0][1]


def montar_item(it, id_compra, tender_items, campos_adicionais):
    item_id = id_compra + it["numeroItem"]          # mesmo id de tender/awards items
    ref = tender_items.get(item_id, {})
    classificacao = dict(ref.get("classification") or {})
    classificacao["id"] = str(it.get("codigoItem"))  # planilha: codigoItem
    classificacao.setdefault("scheme", "CATMAT")

    item = {
        "id": item_id,                                         # numeroItem
        "description": ref.get("description") or it.get("descricaoIitem"),  # descricaoIitem
        "classification": classificacao,                       # codigoItem
        "quantity": it.get("quantidadeItem"),                  # quantidadeItem
        "unit": {
            "name": (ref.get("unit") or {}).get("name"),       # vem do tender (contratos não traz)
            "value": valor(it.get("valorUnitarioItem")),       # valorUnitarioItem
        },
        "relatedLot": ref.get("relatedLot"),                   # extensão Lots
    }
    if campos_adicionais:
        # Linha 100: valorTotalItem -> contracts/items/totalValue (campo proposto)
        item["totalValue"] = valor(it.get("valorTotalItem"))
        # Linha 106: contratoItemExcluido -> contracts/items/sourceDeletion (campo proposto)
        item["sourceDeletion"] = {
            "deleted": it.get("contratoItemExcluido"),
            "date": data_br(it.get("dataHoraExclusaoItem")),
        }
    return item


def montar_milestones(c):
    """
    numeroParcelas -> um contracts/milestones por parcela, type = "payment".
    valorParcela  -> contracts/milestones/value (caminho do OCDS 1.2).
    dueDate não existe na origem.
    """
    n = c.get("numeroParcelas")
    if not n:
        return []
    n = int(n)
    return [
        {
            "id": str(i),
            "title": f"Parcela {i} de {n}",
            "type": "payment",
            "value": valor(c.get("valorParcela")),
        }
        for i in range(1, n + 1)
    ]


def montar_contrato(c, itens, release, id_compra, tender_items, campos_adicionais):
    ini = c.get("dataVigenciaInicial")
    fim = c.get("dataVigenciaFinal")
    duracao = None
    if ini and fim:
        duracao = (datetime.fromisoformat(fim) - datetime.fromisoformat(ini)).days

    contract_items = [montar_item(it, id_compra, tender_items, campos_adicionais) for it in itens]
    descricao = c.get("objeto")
    if c.get("processo"):
        descricao = f"{descricao}\nProcesso administrativo: {c['processo']}"

    contrato = {
        "id": c["numeroControlePncpContrato"],                 # numeroControlePncpContrato
        "awardID": resolver_award_id(release, c["niFornecedor"],
                                     [i["id"] for i in contract_items]),
        "title": f"Contrato {c['numeroContrato']}",            # numeroContrato
        "description": descricao,                              # objeto (+ processo)
        # contracts/status: sem rota definida na planilha (linha 8) -> omitido
        "period": {
            "startDate": data_br(ini),                         # dataVigenciaInicial
            "endDate": data_br(fim),                           # dataVigenciaFinal
            "durationInDays": duracao,                         # calculado
        },
        # valorGlobal; receitaDespesa = "R" -> valor negativo (nota da linha 15).
        # Não soma totalDespesasAcessorias (regra da Data Elements, linha 152).
        "value": valor(c.get("valorGlobal"), negativo=c.get("receitaDespesa") == "R"),
        "items": contract_items,
        # dateSigned: não retornado pela API (linha 39) -> omitido
        # documents: sem fonte nestes endpoints (linha 40) -> omitido
        "milestones": montar_milestones(c),                    # numeroParcelas / valorParcela
    }

    if campos_adicionais:
        # Seção de campos adicionais (linhas 98-107)
        contrato["accumulatedValue"] = valor(c.get("valorAcumulado"))          # linha 98
        contrato["accessoryExpenses"] = valor(c.get("totalDespesasAcessorias"))  # linha 99
        contrato["classification"] = {                                          # linhas 101-102
            "scheme": ESQUEMA_CATEGORIA,
            "id": c.get("codigoCategoria"),
            "description": c.get("nomeCategoria"),
        }
        if c.get("codigoSubcategoria") or c.get("nomeSubcategoria"):              # linhas 103-104
            contrato["subclassification"] = {
                "scheme": ESQUEMA_SUBCATEGORIA,
                "id": c.get("codigoSubcategoria"),
                "description": c.get("nomeSubcategoria"),
            }
        contrato["sourceDeletion"] = {                                           # linhas 105, 107
            "deleted": c.get("contratoExcluido"),
            "date": data_br(c.get("dataHoraExclusao")),
        }
    return contrato


# --------------------------------------------------------------------------
# Recorte por medicamento (opcional)
# --------------------------------------------------------------------------
def recortar_por_item(release, item_id):
    tender = release["tender"]
    tender["items"] = [i for i in tender.get("items", []) if i["id"] == item_id]
    lotes = {i.get("relatedLot") for i in tender["items"]}
    tender["lots"] = [l for l in tender.get("lots", []) if l["id"] in lotes]

    awards = []
    for aw in release.get("awards", []):
        itens = [i for i in aw.get("items", []) if i["id"] == item_id]
        if itens:
            aw["items"] = itens  # award.value permanece o do fornecedor (como na origem)
            awards.append(aw)
    release["awards"] = awards

    contratos = []
    for ct in release.get("contracts", []):
        itens = [i for i in ct.get("items", []) if i["id"] == item_id]
        if itens:
            ct["items"] = itens  # contract.value permanece o valor global do contrato
            contratos.append(ct)
    release["contracts"] = contratos


# --------------------------------------------------------------------------
# Verificações estruturais simples (não substitui o Data Review Tool da OCP)
# --------------------------------------------------------------------------
def verificar(release):
    problemas = []
    for campo in ("ocid", "id", "date", "tag", "initiationType"):
        if not release.get(campo):
            problemas.append(f"release sem {campo}")
    award_ids = {a["id"] for a in release.get("awards", [])}
    tender_item_ids = {i["id"] for i in release.get("tender", {}).get("items", [])}
    party_ids = {p["id"] for p in release.get("parties", [])}
    for ct in release.get("contracts", []):
        if not ct.get("id"):
            problemas.append("contrato sem id")
        if ct.get("awardID") not in award_ids:
            problemas.append(f"{ct.get('id')}: awardID {ct.get('awardID')!r} não existe em awards")
        for it in ct.get("items", []):
            if it["id"] not in tender_item_ids:
                problemas.append(f"{ct['id']}: item {it['id']} não está em tender.items")
        ms = [m["id"] for m in ct.get("milestones", [])]
        if len(ms) != len(set(ms)):
            problemas.append(f"{ct['id']}: milestones com id repetido")
    for aw in release.get("awards", []):
        for s in aw.get("suppliers", []):
            if s["id"] not in party_ids:
                problemas.append(f"award {aw['id']}: fornecedor {s['id']} fora de parties")
    return problemas


# --------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--cnpj", default="00394544000185")
    ap.add_argument("--ano", default="2025")
    ap.add_argument("--sequencial", default="1632")
    ap.add_argument("--codigo-orgao", default="36000", help="código SIASG do órgão (não vem do PNCP)")
    ap.add_argument("--codigo-modalidade", default="05", help="modalidade no Compras (05 = Pregão)")
    ap.add_argument("--item", default="00007", help="numeroItem do medicamento, ou 'todos'")
    ap.add_argument("--sem-campos-adicionais", action="store_true",
                    help="não incluir os campos propostos (linhas 98-107 da planilha)")
    ap.add_argument("--saida", default=str(SAIDA_PADRAO))
    # parse_known_args: no Jupyter o kernel passa argumentos próprios (-f ...).
    args, _ = ap.parse_known_args()
    campos_adicionais = not args.sem_campos_adicionais

    print("1. Compra no PNCP...")
    compra, id_compra = buscar_compra_pncp(args.cnpj, args.ano, args.sequencial)
    data_pub = compra["dataPublicacaoPncp"][:10]
    codigo_ug = compra["unidadeOrgao"]["codigoUnidade"]
    print(f"   idCompra={id_compra}  publicação={data_pub}  UG={codigo_ug}")

    print("2. Release OCDS...")
    pacote, release = buscar_release(args.cnpj, data_pub, id_compra)
    print(f"   ocid={release['ocid']}  itens={len(release['tender'].get('items', []))}  awards={len(release.get('awards', []))}")

    inicio = date.fromisoformat(data_pub)
    hoje = date.today()
    print("3a. Contratos...")
    contratos = buscar_contratos(args.codigo_orgao, codigo_ug, args.codigo_modalidade, id_compra, inicio, hoje)
    print(f"   {len(contratos)} contrato(s)")
    print("3b. Itens de contrato...")
    itens = buscar_itens_contrato(args.codigo_orgao, id_compra, inicio, hoje)
    print(f"   {len(itens)} item(ns)")

    # Regra de publicação: contrato/item excluído na origem não entra.
    contratos = [c for c in contratos if not c.get("contratoExcluido")]
    itens = [i for i in itens if not i.get("contratoExcluido") and not i.get("contratoItemExcluido")]

    # Itens do endpoint 2 sem contrato correspondente no endpoint 1 (ex.: UG diferente)
    conhecidos = {c["numeroControlePncpContrato"] for c in contratos}
    faltando = {i["numeroControlePncpContrato"] for i in itens} - conhecidos
    if faltando:
        print(f"   aviso: itens de contratos não retornados no 1_consultarContratos: {sorted(faltando)}")

    tender_items = {i["id"]: i for i in release["tender"].get("items", [])}
    release["contracts"] = []
    for c in sorted(contratos, key=lambda x: x["dataVigenciaInicial"]):
        its = [i for i in itens if i["numeroControlePncpContrato"] == c["numeroControlePncpContrato"]]
        its.sort(key=lambda i: i["numeroItem"])
        release["contracts"].append(montar_contrato(c, its, release, id_compra, tender_items, campos_adicionais))

    if args.item != "todos":
        recortar_por_item(release, id_compra + args.item.zfill(5))

    # Metadados do release: novo id/data e tag "contract"
    # (só dos contratos que ficaram no documento, depois do recorte)
    mantidos = {ct["id"] for ct in release["contracts"]}
    datas_inclusao = [c["dataHoraInclusao"] for c in contratos
                      if c["numeroControlePncpContrato"] in mantidos and c.get("dataHoraInclusao")]
    if release["contracts"] and datas_inclusao:
        release["date"] = data_br(max(datas_inclusao))
        release["id"] = f"{release['id']}-contratos-{max(datas_inclusao)[:10]}"
        release["tag"] = sorted(set(release.get("tag", [])) | {"contract"},
                                key=["planning", "tender", "award", "contract", "implementation"].index)

    release = limpar(release)

    # Pacote: a API devolve "publisherDTO"; o schema OCDS exige "publisher".
    saida = {
        "uri": pacote.get("uri"),
        "version": "1.1",
        "publishedDate": datetime.now(TZ_BR).isoformat(timespec="seconds"),
        "publisher": pacote.get("publisher") or pacote.get("publisherDTO"),
        "license": pacote.get("license"),
        "publicationPolicy": pacote.get("publicationPolicy"),
        "extensions": [e for e in pacote.get("extensions", []) if "pagination" not in e],
        "releases": [release],
    }
    saida = limpar(saida)

    problemas = verificar(release)
    print("Verificação:", "ok" if not problemas else "")
    for p in problemas:
        print("  -", p)

    Path(args.saida).parent.mkdir(parents=True, exist_ok=True)
    with open(args.saida, "w", encoding="utf-8") as f:
        json.dump(saida, f, ensure_ascii=False, indent=2)
    print(f"Gerado: {args.saida}  (contracts: {len(release.get('contracts', []))})")


if __name__ == "__main__":
    main()

1. Compra no PNCP...
   idCompra=25000505900982025  publicação=2025-07-22  UG=250005
2. Release OCDS...
   ocid=ocds-914jxj-25000505900982025  itens=16  awards=5
3a. Contratos...
   3 contrato(s)
3b. Itens de contrato...
   6 item(ns)
Verificação: ok
Gerado: c:\Users\gabri\OneDrive\Área de Trabalho\joao\TB\medicamentos-fdcc\tasks\mapeamento-ocds\output\exemplo-ocds-contracts.json  (contracts: 1)
